In [1]:
import pandas as pd
import numpy as np
import glob
import os
import warnings

# Silence migration warnings for Pandas 3.0+
warnings.filterwarnings('ignore', category=pd.errors.Pandas4Warning)

# Define the mapping for Benchmark Regions based on BenchmarkRegions.docx
BENCHMARK_MAP = {
    'Sioux Falls, SD': {'SD': ['MINNEHAHA', 'LINCOLN', 'MCCOOK', 'TURNER'], 'MN': ['ROCK']},
    'Billings, MT': {'MT': ['CARBON', 'STILLWATER', 'YELLOWSTONE']},
    'Flagstaff, AZ': {'AZ': ['COCONINO']},
    'Missoula, MT': {'MT': ['MINERAL', 'MISSOULA']},
    'Black Hills, SD (ref)': {
        'SD': ['BUTTE', 'MEADE', 'LAWRENCE', 'PENNINGTON', 'CUSTER', 'OGLALA LAKOTA', 'FALL RIVER']
    }
}

# Define core columns for the analysis-ready dataset (Variable Selection)
CORE_COLUMNS = [
    'assistance_award_unique_key', 'award_id_fain', 'recipient_name',
    'total_obligated_amount', 'total_outlayed_amount', 'total_funding_amount',
    'award_latest_action_date', 'period_of_performance_start_date',
    'primary_place_of_performance_county_name', 'primary_place_of_performance_state_name',
    'awarding_agency_name', 'awarding_sub_agency_name',
    'assistance_type_description', 'cfda_numbers_and_titles'
]

In [2]:
# Adjust base_path to where your 'data' folder is located
base_path = 'data/'
folders = [
    'PrimeAwardSummariesAndSubawards_2026-03-29_benchmark_regions/',
    'PrimeAwardSummariesAndSubawards_2026-03-29_sd/'
]

# Identify all Prime Award Summary files (Assistance and Contracts)
prime_files = []
for folder in folders:
    path = os.path.join(base_path, folder)
    prime_files.extend(glob.glob(path + "*PrimeAwardSummaries*.csv"))

print(f"Found {len(prime_files)} Prime Award files for consolidation.")

Found 4 Prime Award files for consolidation.


In [3]:
frames = []
for f in prime_files:
    # Use low_memory=False to handle the large number of columns in USAspending data
    temp_df = pd.read_csv(f, low_memory=False)
    # Only keep core columns that exist in the current file (Assistance vs Contracts vary slightly)
    existing_cols = [c for c in CORE_COLUMNS if c in temp_df.columns]
    frames.append(temp_df[existing_cols])

# Consolidate all files into one master dataframe
df = pd.concat(frames, ignore_index=True)

# Standardization of date-time
date_cols = [c for c in df.columns if 'date' in c.lower()]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# Financial Cleaning: Replace NaNs with 0.00 as per common assumption for this dataset
fin_cols = ['total_obligated_amount', 'total_outlayed_amount', 'total_funding_amount']
for col in fin_cols:
    if col in df.columns:
        df[col] = df[col].fillna(0)

# Standardize casing for mapping
df['primary_place_of_performance_county_name'] = df['primary_place_of_performance_county_name'].astype(str).str.upper()
df['primary_place_of_performance_state_name'] = df['primary_place_of_performance_state_name'].astype(str).str.upper()

print(f"Consolidated dataframe shape: {df.shape}")

Consolidated dataframe shape: (166622, 14)


In [4]:
def get_benchmark_region(row):
    county = row['primary_place_of_performance_county_name']
    state_name = row['primary_place_of_performance_state_name']
    
    state_to_code = {'SOUTH DAKOTA': 'SD', 'MINNESOTA': 'MN', 'MONTANA': 'MT', 'ARIZONA': 'AZ'}
    state_code = state_to_code.get(state_name)
    
    if not state_code: return 'Other'
    for region, mapping in BENCHMARK_MAP.items():
        if state_code in mapping and county in mapping[state_code]:
            return region
    return 'Other'

# Create the benchmark region tag
df['benchmark_region'] = df.apply(get_benchmark_region, axis=1)

# Filter for only the desired benchmark regions for the final Silver dataset
df_benchmark = df[df['benchmark_region'] != 'Other'].copy()

print("Records per Benchmark Region:")
print(df_benchmark['benchmark_region'].value_counts())

Records per Benchmark Region:
benchmark_region
Sioux Falls, SD          67927
Black Hills, SD (ref)    36365
Billings, MT             27597
Flagstaff, AZ            19748
Missoula, MT             13700
Name: count, dtype: int64


In [ ]:
# 1. Type Hardening: Convert all object/string columns to explicit strings
# This resolves the 'ArrowKeyError' and 'pandas.period' registry conflicts
all_str_cols = df_benchmark.select_dtypes(include=['object', 'string']).columns
for col in all_str_cols:
    df_benchmark[col] = df_benchmark[col].astype(str)

# 2. Define output paths
parquet_out = "processed_data/usaspending_benchmark_awards_consolidated.parquet"
csv_out = "processed_data/usaspending_benchmark_awards_consolidated.csv"

# 3. Export to Parquet (Using fastparquet as primary engine for stability)
try:
    df_benchmark.to_parquet(parquet_out, index=False, engine='fastparquet')
    print(f"Silver Tier Parquet saved: {parquet_out}")
except Exception as e:
    print(f"Fastparquet failed, using pyarrow: {e}")
    df_benchmark.to_parquet(parquet_out, index=False, engine='pyarrow')

# 4. Export to CSV for wider portability
df_benchmark.to_csv(csv_out, index=False, encoding='utf-8')
print(f"Portability CSV saved: {csv_out}")

Fastparquet failed, using pyarrow: Error converting column "recipient_name" to bytes using encoding UTF8. Original error: Unable to avoid copy while creating an array as requested.
Portability CSV saved: usaspending_benchmark_awards_consolidated.csv
